# 03 — Ablations: context size and chunk stratification

Two controlled ablations, both already run and recorded:

- **A. Context size** — does a bigger per-chunk context buy accuracy, and what does it cost?
- **B. Stratification** — is stratified chunking worth its complexity over random chunking?

**This notebook re-runs nothing.** It reads the cached CSVs in `reports/`, so it executes in seconds and is safe to run live in front of a reviewer. The runs themselves took roughly 80 minutes.

Charts are matplotlib; `plotly` is not a project dependency, and adding one for four bar charts would not earn its place in `requirements.txt`.

In [ ]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tabpfn_nids import config

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.spines.top': False,
                     'axes.spines.right': False})

paths = sorted(glob.glob(str(config.REPORTS_DIR / 'ablation_*.csv')))
ablation = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

print(f'{len(paths)} ablation CSV(s), {len(ablation)} runs')
print('source:', ', '.join(Path(p).name for p in paths))
ablation[['label', 'chunk_size', 'stratified', 'n_chunks', 'context_rows',
          'test_rows', 'f1_score', 'roc_auc', 'runtime_seconds']]

## A. Context size

Chunk count is held at 3 and the test set at 1,000 rows, so the only variable is how many training rows each chunk holds. Note that this also changes total training coverage (`n_chunks x chunk_size`), which is the honest way to read it: bigger contexts mean both a richer single context *and* more data overall.

In [ ]:
context = ablation[ablation['stratified'] == True].sort_values('chunk_size')

view = context[['chunk_size', 'context_rows', 'f1_score', 'roc_auc',
                'runtime_seconds']].copy()
view['s_per_1k_test_rows'] = (view['runtime_seconds']
                              / context['test_rows'].values * 1000).round(1)
view.round(4).to_string(index=False)
print(view.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.arange(len(context))
labels = [f"{c:,}" for c in context['chunk_size']]

axes[0].bar(x, context['f1_score'], color='#4C72B0')
axes[0].set_ylim(0.70, 0.80)
axes[0].set_title('F1 vs chunk size')
axes[0].set_ylabel('F1')

axes[1].bar(x, context['roc_auc'], color='#55A868')
axes[1].set_ylim(0.95, 0.965)
axes[1].set_title('ROC-AUC vs chunk size')
axes[1].set_ylabel('ROC-AUC')

axes[2].bar(x, context['runtime_seconds'], color='#C44E52')
axes[2].set_yscale('log')
axes[2].set_title('Runtime vs chunk size (log scale)')
axes[2].set_ylabel('seconds')

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel('chunk size (rows)')

fig.suptitle('Ablation A — per-chunk context size (3 chunks, 1,000 test rows, seed 42)')
fig.tight_layout()
plt.show()

### The cost curve is the finding

Accuracy is close to flat — F1 moves within about 4 points across a 10x change in context size, and ROC-AUC moves by less than 0.006. Runtime is not flat at all: it grows sharply and superlinearly, because attention over the in-context set is quadratic in context length and, on MPS, larger contexts also start paying memory-pressure costs.

The largest context is not even the best F1 here, which is a useful result to be able to state: **on NSL-KDD, paying for a 10,000-row context buys nothing over a 5,000-row one.** With a single seed per point, the accuracy differences are inside the noise floor from notebook 04 — so the defensible claim is "no measurable accuracy gain at large cost", not "smaller is better".

In [ ]:
base = context.iloc[0]
rel = pd.DataFrame({
    'chunk_size': context['chunk_size'].values,
    'F1 delta vs 1,000': (context['f1_score'] - base['f1_score']).values,
    'runtime x vs 1,000': (context['runtime_seconds']
                           / base['runtime_seconds']).values,
}).round(3)
print(rel.to_string(index=False))
print('\nCost of the largest context, per point of F1: '
      f"{(context.iloc[-1]['runtime_seconds'] - base['runtime_seconds']):.0f}s "
      f"extra for {100 * (context.iloc[-1]['f1_score'] - base['f1_score']):+.2f} pp F1.")

## B. Stratification

The control arm for stratified chunking. Both runs use `chunk_size=10,000`, 3 chunks, the same seed and the same test rows; the only difference is whether chunks are built to preserve the population class balance or drawn at random.

In [ ]:
strat = ablation[ablation['chunk_size'] == 10_000].copy()
strat['chunking'] = np.where(strat['stratified'], 'stratified', 'random')

cols = ['chunking', 'max_chunk_balance_drift', 'mean_chunk_confidence',
        'f1_score', 'roc_auc', 'runtime_seconds']
print(strat[cols].to_string(index=False,
      formatters={'max_chunk_balance_drift': '{:.6f}'.format,
                  'mean_chunk_confidence': '{:.4f}'.format,
                  'f1_score': '{:.4f}'.format,
                  'roc_auc': '{:.4f}'.format}))

ratio = (strat.loc[~strat['stratified'], 'max_chunk_balance_drift'].iloc[0]
         / strat.loc[strat['stratified'], 'max_chunk_balance_drift'].iloc[0])
print(f'\nStratification reduces balance drift by {ratio:.0f}x.')
print('\nPer-chunk attack rates')
for _, row in strat.iterrows():
    print(f"  {row['chunking']:<12} {row['chunk_positive_rates']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(strat))
colours = ['#4C72B0', '#DD8452']

axes[0].bar(x, strat['max_chunk_balance_drift'], color=colours)
axes[0].set_title('Class-balance drift across chunks\n(lower is better)')
axes[0].set_ylabel('max - min attack rate')

axes[1].bar(x, strat['f1_score'], color=colours)
axes[1].set_ylim(0.70, 0.80)
axes[1].set_title('F1')
axes[1].set_ylabel('F1')

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(strat['chunking'])

fig.suptitle('Ablation B — stratified vs random chunking '
             '(chunk_size=10,000, 3 chunks, seed 42)')
fig.tight_layout()
plt.show()

### What this one actually shows

Stratification does exactly what it is designed to do to the **chunks**: balance drift drops by more than two orders of magnitude — from about 1.19% to about 0.004%, a 320x reduction. That part is a mechanical guarantee, not a measurement.

What it does *not* show is a corresponding accuracy win — random chunking scored slightly higher F1 on this single seed. The gap is smaller than the seed-to-seed noise floor, so the honest reading is that **on NSL-KDD, stratification is insurance rather than an accuracy lever**: the dataset is close to balanced (about 47% attack), so random chunks are already nearly balanced by luck.

The insurance is still worth buying. On a heavily imbalanced dataset — UNSW-NB15 and CIC-IDS-2018 are far more skewed — a random chunk can end up with too few positives to be informative, and the drift column is what would catch it. One seed on one near-balanced dataset is not grounds for dropping the guarantee.

## Summary of both ablations

In [ ]:
summary = pd.DataFrame([
    {'ablation': 'A. context size',
     'varied': 'chunk_size 1,000 -> 10,000',
     'F1 range': f"{context['f1_score'].min():.4f} - {context['f1_score'].max():.4f}",
     'runtime range': f"{context['runtime_seconds'].min():.0f}s - "
                      f"{context['runtime_seconds'].max():.0f}s",
     'verdict': 'no measurable accuracy gain, 65x cost'},
    {'ablation': 'B. stratification',
     'varied': 'stratified vs random chunks',
     'F1 range': f"{strat['f1_score'].min():.4f} - {strat['f1_score'].max():.4f}",
     'runtime range': f"{strat['runtime_seconds'].min():.0f}s - "
                      f"{strat['runtime_seconds'].max():.0f}s",
     'verdict': 'balance guaranteed; no F1 effect at this skew'},
])
print(summary.to_string(index=False))
print('\nCaveat carried into the report: one seed per configuration. '
      'Differences below the ~2.3 pp F1 noise floor are not effects.')